## Bulut CPU örneğinde Türkçe metin duygu modelinin çıkarım testi (Azure Machine Learning)

M53'teki koçluk modunun metin kanalında kullanılan [`alperengozeten/bert-turkish-emotion`](https://huggingface.co/alperengozeten/bert-turkish-emotion) modelini GPU'suz bir **Azure Machine Learning compute instance** üzerinde çalıştırıp gecikmesini ölçüyorum. Amaç, ucuz bir CPU örneğinde metin kanalının canlı kullanım için yeterince hızlı olup olmadığını görmek. Not defteri SageMaker notebook instance'ta da değişiklik yapmadan çalışır.

> Bu bir **çalışabilirlik ve gecikme** testidir; doğruluk ölçmez. Cümleler sentetiktir, gerçek aday verisi kullanılmaz.

In [ ]:
import json, os, platform, time, urllib.request
from pathlib import Path

def ortam():
    sm = Path("/opt/ml/metadata/resource-metadata.json")
    if sm.exists():
        return "AWS SageMaker", json.loads(sm.read_text()).get("ResourceName", "-"), None
    try:  # Azure Instance Metadata Service (abonelik bilgisi yazdırılmaz)
        req = urllib.request.Request("http://169.254.169.254/metadata/instance/compute?api-version=2021-02-01",
                                     headers={"Metadata": "true"})
        c = json.load(urllib.request.urlopen(req, timeout=2))
        return "Azure Machine Learning", c.get("vmSize"), c.get("location")
    except Exception:
        return "yerel", platform.node(), None

platform_adi, ornek, bolge = ortam()
print("platform      :", platform_adi)
print("örnek / boyut :", ornek)
print("bölge         :", bolge)
print("CPU çekirdeği :", os.cpu_count())
print("RAM           :", round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30, 1), "GB")
print("Python        :", platform.python_version())

In [ ]:
%pip install -q "transformers>=4.51,<5" pandas
try:
    import torch
except ImportError:
    %pip install -q torch --index-url https://download.pytorch.org/whl/cpu

In [ ]:
import torch, pandas as pd
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_ID = "alperengozeten/bert-turkish-emotion"
torch.set_num_threads(os.cpu_count())
t0 = time.perf_counter()
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID).eval()
labels = [model.config.id2label[i] for i in range(model.config.num_labels)]
print(f"yükleme: {time.perf_counter() - t0:.1f} sn | torch {torch.__version__} | etiketler: {labels}")

### Sentetik görüşme cümleleri

In [ ]:
cumleler = [
    "Bu projeyi zamanında teslim ettiğimiz için ekip olarak çok mutluyduk.",
    "Sunucu gece yarısı çöktüğünde açıkçası biraz panikledim.",
    "Yöneticimin geri bildirimi beni başta üzdü ama haklıydı.",
    "Aynı hatanın üçüncü kez tekrarlanması beni gerçekten sinirlendirdi.",
    "Yeni teknolojileri öğrenmek beni her zaman heyecanlandırıyor.",
    "Sonuçları gördüğümde şaşırdım, beklediğimizden çok daha iyiydi.",
    "Günlük toplantılarda sprint durumunu kısaca paylaşıyoruz.",
    "Müşterinin son dakika değişikliği ekipte ciddi bir gerginlik yarattı.",
]

@torch.inference_mode()
def tahmin(metinler):
    enc = tok(metinler, padding=True, truncation=True, max_length=256, return_tensors="pt")
    return model(**enc).logits.softmax(-1)

olas = tahmin(cumleler)
tablo = pd.DataFrame({
    "cümle": cumleler,
    "en olası": [labels[i] for i in olas.argmax(-1)],
    "skor": olas.max(-1).values.numpy().round(3),
    "ikinci": [labels[i] for i in olas.topk(2).indices[:, 1]],
})
pd.set_option("display.max_colwidth", 80)
tablo

### Gecikme ölçümü (tekli ve 8'li toplu çıkarım, 30 tekrar)

In [ ]:
import numpy as np

def olc(metinler, n=30, isinma=3):
    for _ in range(isinma):
        tahmin(metinler)
    sure = []
    for _ in range(n):
        t = time.perf_counter(); tahmin(metinler); sure.append((time.perf_counter() - t) * 1000)
    s = np.array(sure)
    return {"p50 (ms)": round(float(np.percentile(s, 50)), 1), "p95 (ms)": round(float(np.percentile(s, 95)), 1),
            "cümle başına p50 (ms)": round(float(np.percentile(s, 50)) / len(metinler), 1)}

sonuc = pd.DataFrame({"tekli (1 cümle)": olc(cumleler[:1]), "toplu (8 cümle)": olc(cumleler)}).T
sonuc

### Değerlendirme

- M53'te metin kanalı en sık **6 saniyede bir** çalışıyor; ölçülen CPU gecikmesi bu aralığın çok altında kalıyorsa metin modeli için GPU'lu bir örnek gerekmez.
- Toplu çıkarımda cümle başına süre düşüyor; ama canlı görüşmede cümleler tek tek geldiği için belirleyici olan **tekli p95** değeri.
- Skorlar kalibre edilmiş olasılık değildir; model kartında eğitim verisi belgeleri eksik olduğu için sonuçlar işe alım kararında kullanılmaz (bkz. README).
- İş bitince compute instance **durduruldu ve silindi**; açık kalan örnek kredi tüketmeye devam eder.